# Notebook 3 — Extended Linear Regression, Diagnostics and Model Comparison

This notebook builds two models. Model 1 uses a broad set of engineered
features. VIF, correlation, covariance and a heatmap are then used to understand
the data. Model 2 uses a smaller feature set, and both models are compared on
the same unseen test records.

## Use case and target

The goal is to predict `Balance_After_Transaction`. A broader model may capture
more patterns, but unnecessary or highly related features can make a linear
model unstable. A reduced model may generalize better and be easier to explain.

> The dataset has only 50 rows. The notebook is an educational demonstration;
> production modelling would require more historical records and validation.

## 1. Import libraries and read the CSV

Seaborn and Matplotlib create the heatmap. Statsmodels calculates VIF.
Scikit-learn performs preprocessing, modelling and evaluation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

df = pd.read_csv("banking_operations(1).csv")
print("Dataset shape:", df.shape)
df.head()

## 2. Inspect data quality

Missing values are checked before modelling. The date column is converted to a
real datetime so that month, day and day-of-week can be extracted. Transaction
and customer IDs are identifiers, so they are not used as predictive features.

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("Data_Type"))

print("Missing values:")
display(df.isna().sum().to_frame("Missing_Count"))

## 3. Engineer useful date features

Linear regression cannot directly use a date string. The date is therefore
represented using numeric components. These components should be treated as
demonstration features; their real business usefulness depends on longer-term
banking data.

In [ ]:
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])
df["Transaction_Month"] = df["Transaction_Date"].dt.month
df["Transaction_Day"] = df["Transaction_Date"].dt.day
df["Transaction_DayOfWeek"] = df["Transaction_Date"].dt.dayofweek

df[["Transaction_Date", "Transaction_Month", "Transaction_Day", "Transaction_DayOfWeek"]].head()

## 4. Correlation heatmap

Correlation ranges from -1 to +1. Values near +1 indicate a strong positive
linear relationship; values near -1 indicate a strong negative relationship;
values near 0 indicate little linear relationship. Correlation does not prove
causation.

In [ ]:
numeric_columns = [
    "Amount", "Transaction_Month", "Transaction_Day",
    "Transaction_DayOfWeek", "Balance_After_Transaction"
]
correlation_matrix = df[numeric_columns].corr()
display(correlation_matrix)

plt.figure(figsize=(9, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap of Numeric Variables")
plt.tight_layout()
plt.show()

## 5. Covariance matrix

Covariance shows whether two variables tend to move together. Positive values
mean they generally move in the same direction; negative values mean they tend
to move in opposite directions. Its magnitude depends on each variable's scale,
so correlation is usually easier for comparing relationships.

In [ ]:
covariance_matrix = df[numeric_columns].cov()
covariance_matrix

## 6. VIF analysis

Variance Inflation Factor detects multicollinearity among input features.
Common guidance is:

- VIF near 1: little multicollinearity
- VIF from 1 to 5: usually acceptable
- VIF above 5: investigate
- VIF above 10: potentially serious

VIF is calculated on the numeric candidate features. It does not measure a
feature's predictive value; it measures overlap with the other inputs.

In [ ]:
vif_features = ["Amount", "Transaction_Month", "Transaction_Day", "Transaction_DayOfWeek"]
vif_data = df[vif_features].astype(float)

def calculate_vif(data):
    vif_values = []
    for feature in data.columns:
        other_features = data.drop(columns=feature)
        feature_values = data[feature]
        auxiliary_model = LinearRegression()
        auxiliary_model.fit(other_features, feature_values)
        auxiliary_r2 = auxiliary_model.score(other_features, feature_values)
        vif = np.inf if auxiliary_r2 >= 1 else 1 / (1 - auxiliary_r2)
        vif_values.append(vif)
    return pd.DataFrame({"Feature": data.columns, "VIF": vif_values})

vif_table = calculate_vif(vif_data)
vif_table.sort_values("VIF", ascending=False).reset_index(drop=True)

## 7. Define Model 1 features

Model 1 uses numeric, date-derived and categorical information. Missing
categorical values are replaced with the most frequent value. Categories are
converted to one-hot columns, and one category is dropped from each group to
reduce the dummy-variable trap.

In [ ]:
target = "Balance_After_Transaction"

numeric_features_model1 = [
    "Amount", "Transaction_Month", "Transaction_Day", "Transaction_DayOfWeek"
]
categorical_features_model1 = [
    "Account_Type", "Transaction_Type", "Channel", "Branch_City", "Status"
]
feature_columns_model1 = numeric_features_model1 + categorical_features_model1

X_model1 = df[feature_columns_model1]
y = df[target]

train_index, test_index = train_test_split(
    df.index, test_size=0.20, random_state=42
)
X1_train = X_model1.loc[train_index]
X1_test = X_model1.loc[test_index]
y_train = y.loc[train_index]
y_test = y.loc[test_index]

print("Training records:", len(train_index))
print("Testing records :", len(test_index))

## 8. Build and train Model 1

A pipeline prevents information from the test set leaking into preprocessing.
Imputation and one-hot encoding are learned from the training data only.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore")),
])

preprocessor_model1 = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features_model1),
    ("categorical", categorical_pipeline, categorical_features_model1),
])

model1 = Pipeline([
    ("preprocessor", preprocessor_model1),
    ("regressor", LinearRegression()),
])

model1.fit(X1_train, y_train)
pred_model1 = model1.predict(X1_test)
print("Model 1 trained successfully.")

## 9. Predict one record with Model 1

The record contains all Model 1 fields. Categories unseen during training are
handled safely by the encoder.

In [ ]:
single_record_model1 = pd.DataFrame({
    "Amount": [15000],
    "Transaction_Month": [9],
    "Transaction_Day": [21],
    "Transaction_DayOfWeek": [0],
    "Account_Type": ["Savings"],
    "Transaction_Type": ["Transfer"],
    "Channel": ["UPI"],
    "Branch_City": ["Hyderabad"],
    "Status": ["Completed"],
})

single_prediction_model1 = model1.predict(single_record_model1)[0]
print(f"Model 1 predicted balance: ₹{single_prediction_model1:,.2f}")

## 10. Define and train reduced Model 2

Model 2 keeps only `Amount` and `Transaction_Month`. This smaller specification
is chosen because these numeric features provide a compact, interpretable
baseline while avoiding the large number of one-hot columns created from only
50 records. Feature selection should ultimately be validated with more data.

In [ ]:
selected_features_model2 = ["Amount", "Transaction_Month"]
X_model2 = df[selected_features_model2]

X2_train = X_model2.loc[train_index]
X2_test = X_model2.loc[test_index]

model2 = LinearRegression()
model2.fit(X2_train, y_train)
pred_model2 = model2.predict(X2_test)

single_record_model2 = pd.DataFrame({
    "Amount": [15000],
    "Transaction_Month": [9],
})
single_prediction_model2 = model2.predict(single_record_model2)[0]
print(f"Model 2 predicted balance: ₹{single_prediction_model2:,.2f}")

## 11. Evaluation function

Both models are evaluated on exactly the same test rows.

- Lower MAE and RMSE are better.
- Higher R² is better.
- Adjusted R² penalizes adding many predictors and is useful when comparing
  models of different sizes. With very small test samples and many encoded
  features, it may be undefined; the notebook reports this safely.

In [ ]:
def adjusted_r2(r2_value, sample_count, predictor_count):
    denominator = sample_count - predictor_count - 1
    if denominator <= 0:
        return np.nan
    return 1 - (1 - r2_value) * (sample_count - 1) / denominator


def evaluate_model(name, actual, predicted, predictor_count):
    mae = mean_absolute_error(actual, predicted)
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, predicted)
    adj_r2 = adjusted_r2(r2, len(actual), predictor_count)
    return {
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R²": r2,
        "Adjusted_R²": adj_r2,
        "Predictor_Count": predictor_count,
    }

## 12. Compare Model 1 and Model 2

The predictor count for Model 1 is measured after one-hot encoding. This makes
the complexity difference between the models explicit.

In [ ]:
model1_predictor_count = model1.named_steps["preprocessor"].transform(X1_train).shape[1]
model2_predictor_count = X2_train.shape[1]

comparison = pd.DataFrame([
    evaluate_model("Model 1: Full Feature Set", y_test, pred_model1, model1_predictor_count),
    evaluate_model("Model 2: Reduced Feature Set", y_test, pred_model2, model2_predictor_count),
])
comparison

## 13. Automatically identify the better test model

The decision below prioritizes RMSE because it remains in balance units and
penalizes large errors. R² and model complexity are also displayed so the
choice is not based on one number alone.

In [ ]:
best_row = comparison.loc[comparison["RMSE"].idxmin()]
other_row = comparison.loc[comparison["RMSE"].idxmax()]

print(f"Better model on this test split: {best_row['Model']}")
print(f"Reason: its RMSE is lower by ₹{other_row['RMSE'] - best_row['RMSE']:,.2f}.")

if best_row["Model"].startswith("Model 2"):
    print("The reduced model generalizes better here and is easier to explain.")
    print("The full model likely has too many encoded predictors for only 40 training rows.")
else:
    print("The additional engineered and categorical features improve test predictions here.")
    print("Still, validate this result with more data because the test set contains only 10 rows.")

## 14. Compare individual test predictions

This table helps identify records where one model makes a noticeably larger
error than the other.

In [ ]:
prediction_comparison = pd.DataFrame({
    "Actual_Balance": y_test.values,
    "Model_1_Prediction": pred_model1,
    "Model_2_Prediction": pred_model2,
})
prediction_comparison["Model_1_Absolute_Error"] = (
    prediction_comparison["Actual_Balance"] - prediction_comparison["Model_1_Prediction"]
).abs()
prediction_comparison["Model_2_Absolute_Error"] = (
    prediction_comparison["Actual_Balance"] - prediction_comparison["Model_2_Prediction"]
).abs()
prediction_comparison.round(2)

## Final interpretation

1. Correlation shows linear relationships with the target.
2. Covariance shows whether variables move together, but its size depends on scale.
3. VIF highlights overlapping information among numeric inputs.
4. The full model has more flexibility, but a small dataset makes overfitting likely.
5. The reduced model is simpler and may generalize better.
6. The final model choice should be based on unseen-data metrics, business logic,
   stability across multiple validation splits, and a larger dataset—not training
   performance alone.

## Save both trained models as pickle

Run this cell after both models are trained. Each `.pkl` file is saved in the notebook’s current working directory. Only load pickle files from trusted sources.

In [ ]:
import pickle

with open("linear_regression_model1.pkl", "wb") as file:
    pickle.dump(model1, file)
print("Saved linear_regression_model1.pkl")

with open("linear_regression_model2.pkl", "wb") as file:
    pickle.dump(model2, file)
print("Saved linear_regression_model2.pkl")
